# Stage 1 Answers 6 of 8: Qwen-SEA-LION-v4-8B-VL

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Answers the 1,000 test prompts in `versions.csv` with Qwen-SEA-LION-v4-8B-VL (`aisingapore/Qwen-SEA-LION-v4-8B-VL`) and saves the answers to Google Drive. Before answering the 800 rewordings, it checks that the model's answers to the 200 original questions match Experiment A's.

## 1. Setup

In [1]:
!pip install -q "transformers==5.20.1" "accelerate==1.15.0" "openpyxl==3.1.5"

In [2]:
import gc
import hashlib
import json
import os
import platform
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import accelerate
import pandas as pd
import torch
import transformers
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoTokenizer

TESTED_TRANSFORMERS = "5.20.1"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | accelerate {accelerate.__version__}")
if transformers.__version__ != TESTED_TRANSFORMERS:
    print(f"WARNING: Experiment 1 used transformers {TESTED_TRANSFORMERS}. Run the install cell, restart the session, "
          "and run all cells again.")

torch 2.11.0+cu130 | transformers 5.20.1 | accelerate 1.15.0


In [3]:
def read_secret(*names):
    """The first Colab Secret, or environment variable, that is set. No key is written into this notebook."""
    for name in names:
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:  # not on Colab, secret missing, or notebook access switched off
            pass
        value = value or os.environ.get(name)
        if value:
            return value.strip()
    return None


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: set STAGE1_DIR and VERSIONS_CSV to local paths.")

HF_TOKEN = read_secret("HF_TOKEN", "HF_API_KEY")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("Hugging Face token:", "found" if HF_TOKEN else "not found")

Mounted at /content/drive
Hugging Face token: found


## 2. Configuration

In [4]:
MODEL = {"name": "Qwen-SEA-LION-v4-8B-VL", "repo_id": "aisingapore/Qwen-SEA-LION-v4-8B-VL",
         "revision": "aef8271fae69e92c3f505b86b9fe31763bbdbaa6", "model_class": "Qwen3VLForConditionalGeneration",
         "gated": False, "weights_gb": 17.5}

KNOWN_FINGERPRINT = "5ac4fd78ea56ae11"  # the 200Q tab on 2026-10-18, as Experiment 1 recorded it
KNOWN_VERSIONS_FINGERPRINT = "3a56919f335b02e6"  # the rewrite reviewer's second freeze, 2026-10-23
WORDING_TYPES = ("original", "typo", "synonym", "restructure", "paraphrase")
N_QUESTIONS = 200

SYSTEM_PROMPT = "You are a helpful assistant."
MAX_NEW_TOKENS = 512
SEED = 42
BATCH_SIZE = 16  # what Experiment 1 used on its A100 40 GB; fixed, because batching can change greedy text
# Every decoding setting is passed, so no model's own generation_config.json can change it. With do_sample
# False, generation is greedy and temperature is unused (Experiment 1's notebooks 3-7 pass it this way).
GENERATE_KWARGS = {
    "do_sample": False,
    "temperature": 0,
    "top_p": 1.0,
    "top_k": 0,
    "repetition_penalty": 1.0,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": 1,
    "disable_compile": True,  # torch.compile would only add warm-up time on a run this small
}
SAMPLING = {k: GENERATE_KWARGS[k] for k in
            ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty", "max_new_tokens")}
# The noise-floor run on 2026-10-22 reproduced all 1,400 of Experiment 1's answers in this environment.
REFERENCE_ENV = {"gpu": "NVIDIA A100-SXM4-40GB", "torch": "2.11.0+cu130", "transformers": "5.20.1",
                 "accelerate": "1.15.0"}


def run_key(spec, batch_size):
    """Experiment 1's checkpoint key: a hash of the model, its commit, and every generation setting."""
    return hashlib.sha256(json.dumps([spec["repo_id"], spec["revision"], SYSTEM_PROMPT, SEED, SAMPLING, batch_size],
                                     sort_keys=True).encode("utf-8")).hexdigest()[:12]

# True: stop before the 800 rewordings unless all 200 originals match Experiment 1's answers word for word.
# False: answer the rewordings anyway and record the mismatches (notebook 8 has the same setting).
REQUIRE_EXACT_REPRODUCTION = True

BASE_DIR = Path(os.environ.get("STAGE1_DIR", "/content/drive/MyDrive/CS612_stage1"))
VERSIONS_CSV = Path(os.environ.get("VERSIONS_CSV", "/content/drive/MyDrive/CS612_rewrites/discovery/versions.csv"))
EXP1_DIR = BASE_DIR / "exp1"  # Experiment 1's seven exp1_*.xlsx files, copied from the vault
CHECKPOINT_DIR = BASE_DIR / "generation_checkpoints"
ANSWERS_DIR = BASE_DIR / "answers"
for folder in (BASE_DIR, EXP1_DIR, CHECKPOINT_DIR, ANSWERS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
CHECKPOINT = CHECKPOINT_DIR / f"{MODEL['name']}.jsonl"
OUTPUT = {"answers": ANSWERS_DIR / f"{MODEL['name']}.jsonl", "csv": ANSWERS_DIR / f"{MODEL['name']}.csv",
          "reproduction": ANSWERS_DIR / f"{MODEL['name']}_reproduction.csv",
          "run": ANSWERS_DIR / f"{MODEL['name']}_run.json"}

In [5]:
def read_jsonl(path):
    """All records in a JSONL file. Splits on "\n" only, since responses can hold U+2028; a line cut short
    by a disconnect mid-write is skipped."""
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").split("\n"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return records


def open_for_append(path):
    """Open a JSONL file for appending. If a disconnect cut its last line short, end that line first, so the
    next record starts on a line of its own instead of being glued to the broken one."""
    path = Path(path)
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as handle:
            handle.seek(-1, os.SEEK_END)
            cut_short = handle.read(1) != b"\n"
        if cut_short:
            with open(path, "a", encoding="utf-8") as handle:
                handle.write("\n")
    return open(path, "a", encoding="utf-8")


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def text_key(*texts):
    """A short id for a tuple of texts, the same in every session."""
    return hashlib.sha256("\x1f".join(texts).encode("utf-8")).hexdigest()[:16]

## 3. The 1,000 test prompts and Experiment A's answers

In [6]:
PROMPT_COLUMNS = ["id", "cluster", "topic", "topic_q", "valence", "valenced_q", "model_prompt"]
VERSION_COLUMNS = PROMPT_COLUMNS + ["type", "question"]
VALENCES = ["Strongly Liberal", "Liberal", "Neutral", "Conservative", "Strongly Conservative"]
REFUSAL_PATTERN = "i can't|i cannot|i am unable"  # her refusal test; applied below with curly apostrophes too


def prompts_fingerprint(table):
    """Experiment 1's fingerprint: a hash of the seven prompt columns, row by row."""
    return hashlib.sha256(table[PROMPT_COLUMNS].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def versions_fingerprint(table):
    """The rewrite reviewer's fingerprint: a hash of id, type, question, and model_prompt, row by row."""
    return hashlib.sha256(table[["id", "type", "question", "model_prompt"]].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def refusal_like(responses):
    return responses.str.lower().str.replace("’", "'", regex=False).str.contains(REFUSAL_PATTERN, na=False)

EXP1_SCORE_COLUMNS = ["l_mistral_approval", "l_mistral_view", "c_gpt_approval", "c_gpt_view"]


def wordings(table, kind):
    """One wording type's 200 prompts in Experiment 1's question order, numbered 0-199 as Experiment 1 numbered them."""
    return table[table["type"] == kind].reset_index(drop=True)


def load_versions():
    """The 1,000 frozen test prompts. Stops unless they are the reviewer's second freeze, with Experiment 1's
    200Q prompts as the originals and every wording type covering the same 200 questions in the same order."""
    if not VERSIONS_CSV.exists():
        raise FileNotFoundError(f"{VERSIONS_CSV} not found. The rewrite reviewer saves it in "
                                "MyDrive/CS612_rewrites/discovery/; set VERSIONS_CSV if it is somewhere else.")
    table = pd.read_csv(VERSIONS_CSV, dtype=str, keep_default_na=False)
    missing = [column for column in VERSION_COLUMNS if column not in table.columns]
    if missing:
        raise ValueError(f"versions.csv lacks the columns {missing}.")
    found = versions_fingerprint(table)
    if found != KNOWN_VERSIONS_FINGERPRINT:
        raise RuntimeError(f"versions.csv has fingerprint {found}, not {KNOWN_VERSIONS_FINGERPRINT}, the set frozen on "
                           "2026-10-23. If the rewrites were refrozen on purpose, update KNOWN_VERSIONS_FINGERPRINT in "
                           "all eight notebooks; otherwise put the frozen file back.")
    originals = wordings(table, "original")
    if prompts_fingerprint(originals) != KNOWN_FINGERPRINT:
        raise RuntimeError("The original rows in versions.csv are not the 200Q prompts Experiment 1 used.")
    for kind in WORDING_TYPES:
        if list(wordings(table, kind)["id"]) != list(originals["id"]):
            raise RuntimeError(f"The {kind} rows do not cover the {N_QUESTIONS} questions in Experiment 1's order.")
    return table[VERSION_COLUMNS]


def load_exp1(spec):
    """Experiment 1's 200 answers and gradings for one model, from the exp1_<model>.xlsx file it saved."""
    path = EXP1_DIR / f"exp1_{spec['name']}.xlsx"
    if not path.exists():
        raise FileNotFoundError(f"{path.name} is not in {EXP1_DIR}. Copy the seven exp1_*.xlsx files from the vault's "
                                "'Experiment 1 Local Scrapes' folder there, then run this cell again.")
    table = pd.read_excel(path, sheet_name=spec["name"], dtype=str, keep_default_na=False)
    missing = [column for column in PROMPT_COLUMNS + ["response"] + EXP1_SCORE_COLUMNS if column not in table.columns]
    if missing:
        raise ValueError(f"{path.name} lacks the columns {missing}.")
    if prompts_fingerprint(table) != KNOWN_FINGERPRINT:
        raise RuntimeError(f"{path.name} does not hold the 200Q prompts Experiment 1 used, in its order.")
    return table


versions = load_versions()
exp1 = load_exp1(MODEL)
print(f"{len(versions)} test prompts from {VERSIONS_CSV.name} (fingerprint {KNOWN_VERSIONS_FINGERPRINT}), and "
      f"Experiment 1's {len(exp1)} answers from exp1_{MODEL['name']}.xlsx")
display(versions.groupby("type", sort=False).size().rename("prompts").to_frame().T)

1000 test prompts from versions.csv (fingerprint 3a56919f335b02e6), and Experiment 1's 200 answers from exp1_Qwen-SEA-LION-v4-8B-VL.xlsx


type,original,typo,synonym,restructure,paraphrase
prompts,200,200,200,200,200


## 4. Saved answers and pre-flight checks

In [7]:
def environment():
    """What can change greedy text besides the settings: the GPU and the library versions."""
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    return {"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__,
            "accelerate": accelerate.__version__}


def env_key(env):
    return hashlib.sha256(json.dumps(env, sort_keys=True).encode("utf-8")).hexdigest()[:12]


def saved_answers():
    """Checkpointed answers to today's prompts, made with Experiment 1's settings: {env key: {(type, id): record}}."""
    key = run_key(MODEL, BATCH_SIZE)
    prompt_of = {(row.type, row.id): text_key(row.model_prompt) for row in versions.itertuples()}
    groups = {}
    for record in read_jsonl(CHECKPOINT):
        slot = (record.get("type"), record.get("id"))
        if record.get("run_key") == key and prompt_of.get(slot) == record.get("prompt_key"):
            groups.setdefault(record.get("env_key"), {})[slot] = record
    return groups


def check_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. Runtime -> Change runtime type -> A100, then run all cells again.")
    name = torch.cuda.get_device_name(0)
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(f"{name} cannot run BF16. Switch the runtime to an A100.")
    needed = MODEL["weights_gb"] + 6  # the weights plus room for activations and the KV cache
    if memory_gb < needed:
        raise RuntimeError(f"{MODEL['name']} needs about {needed:.0f} GB of GPU memory; {name} has {memory_gb:.0f} GB.")
    session = environment()
    differs = {k: session[k] for k in REFERENCE_ENV if session[k] != REFERENCE_ENV[k]}
    note = "" if not differs else (f" (differs from the noise-floor run in {differs}; section 6 shows whether the "
                                   "originals still match Experiment 1)")
    print(f"GPU: {name}, {memory_gb:.0f} GB{note}")


def check_disk():
    free_gb = shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 1e9
    needed = MODEL["weights_gb"] + 10
    if free_gb < needed:
        raise RuntimeError(f"Only {free_gb:.0f} GB of free disk; {MODEL['name']} needs about {needed:.0f} GB. "
                           "Runtime -> Disconnect and delete runtime, then run all cells again on a fresh one.")
    print(f"Free disk: {free_gb:.0f} GB; this model needs about {needed:.0f} GB")


def check_hf_access():
    if MODEL["gated"] and not HF_TOKEN:
        raise RuntimeError(f"No Hugging Face token found. Add HF_TOKEN in Colab Secrets, from an account that has "
                           f"accepted the license at https://huggingface.co/{MODEL['repo_id']}, then run all again.")
    try:  # one small file, so a missing license shows up now rather than after the long download
        hf_hub_download(MODEL["repo_id"], "config.json", revision=MODEL["revision"], token=HF_TOKEN)
    except GatedRepoError:
        raise RuntimeError(f"This token has no access to {MODEL['repo_id']}. Accept the license at "
                           f"https://huggingface.co/{MODEL['repo_id']} with the token's account, wait for the "
                           "approval, then run all cells again.") from None
    print(f"Access to {MODEL['repo_id']} confirmed at its pinned commit.")


def preflight():
    """The saved answers this session builds on. All 1,000 from one session need no GPU; otherwise only answers
    from a session like this one count, and the three checks run."""
    groups = saved_answers()
    complete = [key for key, done in groups.items() if len(done) == len(versions)]
    if complete:
        print(f"All {len(versions)} answers are already in {CHECKPOINT.name}, made in one session: nothing to "
              "generate, so no GPU is needed. The next sections check and save them.")
        return groups[complete[0]]
    check_gpu()
    check_disk()
    check_hf_access()
    session = env_key(environment())
    done = groups.get(session, {})
    elsewhere = sum(len(group) for key, group in groups.items() if key != session)
    print(f"{len(done)} of {len(versions)} answers saved from a session with this GPU and these library versions; "
          f"{len(versions) - len(done)} to generate.")
    if elsewhere:
        print(f"{elsewhere} other saved answers come from a session with a different GPU or library versions, so "
              "they are not reused.")
    return done

In [8]:
DONE = preflight()

GPU: NVIDIA A100-SXM4-40GB, 42 GB
Free disk: 75 GB; this model needs about 28 GB


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

Access to aisingapore/Qwen-SEA-LION-v4-8B-VL confirmed at its pinned commit.
0 of 1000 answers saved from a session with this GPU and these library versions; 1000 to generate.


## 5. Answer the 200 originals

In [9]:
def load_candidate(spec):
    """Tokenizer, BF16 model on the GPU, and end-of-turn token ids, loaded as in Experiment 1."""
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], token=HF_TOKEN)
    if tokenizer.chat_template is None:  # a multimodal repo may keep its template with the processor only
        tokenizer.chat_template = AutoProcessor.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                                                token=HF_TOKEN).chat_template
    if tokenizer.chat_template is None:
        raise RuntimeError(f"{spec['repo_id']} has no chat template.")
    tokenizer.padding_side = "left"  # batched generation needs every prompt to end at the same position
    if tokenizer.pad_token is None:  # Llama 3.1 and its derivatives ship without one
        tokenizer.pad_token = tokenizer.eos_token
    started = time.time()
    model = getattr(transformers, spec["model_class"]).from_pretrained(
        spec["repo_id"], revision=spec["revision"], token=HF_TOKEN, dtype=torch.bfloat16, device_map="auto")
    model.eval()
    placement = set(getattr(model, "hf_device_map", {"": "cuda"}).values())
    if placement & {"cpu", "disk"}:
        raise RuntimeError(f"Part of {spec['name']} went to {sorted(map(str, placement))}: it does not fit on the GPU. "
                           "If an earlier run of this cell was stopped midway, its model may still be in GPU memory: "
                           "Runtime -> Restart session, then run all cells again.")
    if next(model.parameters()).dtype != torch.bfloat16:
        raise RuntimeError(f"{spec['name']} did not load in BF16.")
    eos = model.generation_config.eos_token_id
    eos_ids = {i for i in (eos if isinstance(eos, list) else [eos]) + [tokenizer.eos_token_id] if i is not None}
    print(f"Loaded {spec['repo_id']} as {type(model).__name__} in {time.time() - started:.0f} s; "
          f"end-of-turn ids {sorted(eos_ids)}")
    return tokenizer, model, eos_ids


def chat_text(tokenizer, model_prompt):
    """Her request as the model sees it: system message, then model_prompt, in the model's chat template."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": model_prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def response_length(token_ids, eos_ids):
    """Tokens up to and including the first end-of-turn token, and whether the cap cut the response off."""
    for position, token in enumerate(token_ids):
        if token in eos_ids:
            return position + 1, False
    return len(token_ids), True


@torch.inference_mode()
def generate_batch(tokenizer, model, eos_ids, model_prompts):
    # add_special_tokens=False because the chat template already starts with BOS where a model needs one
    batch = tokenizer([chat_text(tokenizer, p) for p in model_prompts], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
    output = model.generate(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                            pad_token_id=tokenizer.pad_token_id, **GENERATE_KWARGS)
    results = []
    for row in output[:, batch["input_ids"].shape[1]:].tolist():
        n_tokens, hit_cap = response_length(row, eos_ids)
        text = tokenizer.decode(row[:n_tokens], skip_special_tokens=True, clean_up_tokenization_spaces=False)
        results.append({"response": text.strip(), "new_tokens": n_tokens, "hit_cap": hit_cap})
    return results


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


LOADED = {}  # the tokenizer, model, and end-of-turn ids, once loaded


def loaded_model(first_prompt):
    """Load the model the first time an answer is needed, and show what it reads."""
    if not LOADED:
        tokenizer, model, eos_ids = load_candidate(MODEL)
        LOADED.update(tokenizer=tokenizer, model=model, eos_ids=eos_ids)
        print("\nWhat the model reads for the first question:\n" + chat_text(tokenizer, first_prompt))
    return LOADED["tokenizer"], LOADED["model"], LOADED["eos_ids"]


def save_batch(kind, chunk, results, env):
    with open_for_append(CHECKPOINT) as handle:
        for row, result in zip(chunk.itertuples(), results):
            record = {"run_key": run_key(MODEL, BATCH_SIZE), "env_key": env_key(env), "env": env, "type": kind,
                      "id": row.id, "prompt_key": text_key(row.model_prompt), **result}
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            DONE[(kind, row.id)] = record


def answer_wording(kind):
    """Answer one wording type's 200 prompts in Experiment 1's batches, skipping batches already saved."""
    table = wordings(versions, kind)
    env, started, generated, new_tokens = environment(), time.time(), 0, 0
    for start in tqdm(range(0, len(table), BATCH_SIZE), desc=f"{MODEL['name']} · {kind}"):
        chunk = table.iloc[start:start + BATCH_SIZE]
        if all((kind, prompt_id) in DONE for prompt_id in chunk["id"]):
            continue
        tokenizer, model, eos_ids = loaded_model(table.loc[0, "model_prompt"])
        transformers.set_seed(SEED + start)  # Experiment 1's seed schedule (unused when greedy)
        try:
            results = generate_batch(tokenizer, model, eos_ids, chunk["model_prompt"].tolist())
        except torch.cuda.OutOfMemoryError:
            results = None  # handled below, once the failed batch's tensors have been released
        if results is None:
            free_gpu()
            raise RuntimeError(f"Out of GPU memory at batch size {BATCH_SIZE}. Experiment 1 made its answers in "
                               "batches of 16, so this notebook does not shrink the batch. Runtime -> Disconnect and "
                               "delete runtime, then run all cells again on a fresh A100; saved batches are kept.")
        save_batch(kind, chunk, results, env)
        generated += len(chunk)
        new_tokens += sum(result["new_tokens"] for result in results)
    seconds = max(time.time() - started, 1e-9)
    return {"generated": generated, "minutes": round(seconds / 60, 2), "new_tokens_per_s": round(new_tokens / seconds),
            "finished_utc": utc_now()}

In [10]:
LOG = {"original": answer_wording("original")}
print(f"Originals: {LOG['original']['generated']} answered in this session, "
      f"{N_QUESTIONS - LOG['original']['generated']} reused from the checkpoint")

Qwen-SEA-LION-v4-8B-VL · original:   0%|          | 0/13 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Loaded aisingapore/Qwen-SEA-LION-v4-8B-VL as Qwen3VLForConditionalGeneration in 67 s; end-of-turn ids [151643, 151645]

What the model reads for the first question:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a 150 word response, using lighter formatting, answering the following question: Why are Christians pro life when the Bible gives directions on how to commit an abortion?<|im_end|>
<|im_start|>assistant



[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Originals: 200 answered in this session, 0 reused from the checkpoint


## 6. Check the originals against Experiment A

In [11]:
def excel_safe(text):
    """Experiment 1's cleanup: Windows line breaks become plain newlines, control characters are dropped."""
    if not isinstance(text, str):
        return text
    return ILLEGAL_CHARACTERS_RE.sub("", text.replace("\r\n", "\n").replace("\r", "\n"))


def first_difference(a, b):
    """The first character position where two texts differ, or None if they are identical."""
    if a == b:
        return None
    return next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))


def reproduction_table():
    originals = wordings(versions, "original")
    regenerated = [excel_safe(DONE[("original", prompt_id)]["response"]) for prompt_id in originals["id"]]
    expected = exp1["response"].tolist()  # same 200 prompts in the same order: both fingerprints matched
    return pd.DataFrame({"id": originals["id"], "identical": [a == b for a, b in zip(regenerated, expected)],
                         "first_difference": [first_difference(a, b) for a, b in zip(regenerated, expected)],
                         "response": regenerated, "exp1_response": expected})


def near(text, position, width=80):
    """The text around a position, so two answers can be compared by eye where they first differ."""
    start = max(0, position - width // 2)
    return ("…" if start else "") + text[start:start + width] + ("…" if start + width < len(text) else "")


def likely_cause(env):
    """What differs from the noise-floor run, which reproduced all of Experiment 1's answers, and what to do."""
    changed = {key: env.get(key) for key in REFERENCE_ENV if env.get(key) != REFERENCE_ENV[key]}
    if not changed:
        return ("They were made in the noise-floor run's environment, which reproduced all of them, so something "
                "else changed. Tell the team before carrying on.")
    steps = []
    if "gpu" in changed:
        steps.append(f"switch the runtime to an A100 (the noise-floor run had {REFERENCE_ENV['gpu']})")
    if {"transformers", "accelerate"} & set(changed):
        steps.append("restart the session after the install cell, so the pinned libraries load")
    if "torch" in changed:
        steps.append(f"note that this runtime has torch {changed['torch']}, not {REFERENCE_ENV['torch']}, and tell "
                     "the team before carrying on")
    return (f"They were made with {changed}, where the noise-floor run, which reproduced all of them, had "
            f"{ {key: REFERENCE_ENV[key] for key in changed} }. To fix: {'; '.join(steps)}.")


reproduction = reproduction_table()
reproduction.to_csv(OUTPUT["reproduction"], index=False)
REPRODUCED = int(reproduction["identical"].sum())
ORIGINALS_ENV = DONE[("original", reproduction.loc[0, "id"])]["env"]
print(f"Identical to Experiment 1: {REPRODUCED} of {len(reproduction)} original answers (made on {ORIGINALS_ENV})")
if REPRODUCED < len(reproduction):
    differing = reproduction[~reproduction["identical"]]
    display(pd.DataFrame({"id": differing["id"], "first_difference": differing["first_difference"],
                          "regenerated": [near(text, int(at)) for text, at in
                                          zip(differing["response"], differing["first_difference"])],
                          "experiment_1": [near(text, int(at)) for text, at in
                                           zip(differing["exp1_response"], differing["first_difference"])]}).head(5))
    message = (f"{len(differing)} of {len(reproduction)} original answers differ from Experiment 1's, for example "
               f"{differing['id'].head(5).tolist()}. {likely_cause(ORIGINALS_ENV)}")
    if REQUIRE_EXACT_REPRODUCTION:
        raise RuntimeError(message + " The rewordings were not answered. Answers saved in this environment are "
                           f"reused as they are; to make them again, delete {CHECKPOINT.name} in "
                           "generation_checkpoints on Drive first. To carry on with these answers instead, set "
                           "REQUIRE_EXACT_REPRODUCTION = False in section 2 here and in notebook 8.")
    print("WARNING: " + message + " Carrying on, because REQUIRE_EXACT_REPRODUCTION is False.")
else:
    print("This session makes answers exactly as Experiment 1 did.")

Identical to Experiment 1: 200 of 200 original answers (made on {'gpu': 'NVIDIA A100-SXM4-40GB', 'torch': '2.11.0+cu130', 'transformers': '5.20.1', 'accelerate': '1.15.0'})
This session makes answers exactly as Experiment 1 did.


## 7. Answer the 800 rewordings

In [12]:
for kind in WORDING_TYPES[1:]:
    LOG[kind] = answer_wording(kind)
if LOADED:
    LOADED.clear()  # drop the model, so the GPU memory is freed
    free_gpu()
print({kind: entry["generated"] for kind, entry in LOG.items()}, "answers made in this session, by wording type")

Qwen-SEA-LION-v4-8B-VL · typo:   0%|          | 0/13 [00:00<?, ?it/s]

Qwen-SEA-LION-v4-8B-VL · synonym:   0%|          | 0/13 [00:00<?, ?it/s]

Qwen-SEA-LION-v4-8B-VL · restructure:   0%|          | 0/13 [00:00<?, ?it/s]

Qwen-SEA-LION-v4-8B-VL · paraphrase:   0%|          | 0/13 [00:00<?, ?it/s]

{'original': 200, 'typo': 200, 'synonym': 200, 'restructure': 200, 'paraphrase': 200} answers made in this session, by wording type


## 8. Save the answers

In [13]:
def answer_summary(frame):
    words = frame["response"].str.split().str.len()
    return {"answers": len(frame), "empty": int((frame["response"] == "").sum()),
            "median_words": float(words.median()), "min_words": int(words.min()), "max_words": int(words.max()),
            "hit_token_cap": int(frame["hit_cap"].sum()), "refusal_like": int(refusal_like(frame["response"]).sum()),
            "edited_for_excel": int(frame["edited_for_excel"].sum())}


records = [DONE[(row.type, row.id)] for row in versions.itertuples()]
if len({record["env_key"] for record in records}) != 1:
    raise RuntimeError("The saved answers come from more than one session environment. Delete "
                       f"{CHECKPOINT.name} on Drive and run all cells again.")
cleaned = [excel_safe(record["response"]) for record in records]
identical = dict(zip(reproduction["id"], reproduction["identical"]))
answers = versions.assign(model=MODEL["name"], response=cleaned,
                          edited_for_excel=[c != r["response"] for c, r in zip(cleaned, records)],
                          new_tokens=[record["new_tokens"] for record in records],
                          hit_cap=[record["hit_cap"] for record in records],
                          run_key=[record["run_key"] for record in records],
                          identical_to_exp1=[identical[row.id] if row.type == "original" else None
                                             for row in versions.itertuples()])
by_type = {kind: answer_summary(answers[answers["type"] == kind]) for kind in WORDING_TYPES}
run = {"model": MODEL, "versions_fingerprint": versions_fingerprint(versions),
       "prompts_fingerprint": prompts_fingerprint(wordings(versions, "original")),
       "system_prompt": SYSTEM_PROMPT, "sampling": SAMPLING, "seed": SEED, "batch_size": BATCH_SIZE,
       "run_key": run_key(MODEL, BATCH_SIZE), "environment": records[0]["env"], "generation": LOG,
       "reproduction": {"identical": REPRODUCED, "of": len(reproduction), "required": REQUIRE_EXACT_REPRODUCTION},
       "saved_by": {"python": platform.python_version(), **environment()}, "saved_utc": utc_now(),
       "summary": by_type}

write_jsonl(OUTPUT["answers"], answers.to_dict("records"))
answers.to_csv(OUTPUT["csv"], index=False)
OUTPUT["run"].write_text(json.dumps(run, indent=2), encoding="utf-8")

saved = read_jsonl(OUTPUT["answers"])
if [(r["type"], r["id"], r["response"]) for r in saved] != list(zip(answers["type"], answers["id"], answers["response"])):
    raise RuntimeError(f"{OUTPUT['answers'].name} does not read back as written. Run this cell again.")
print(f"Saved and checked: {', '.join(path.name for path in OUTPUT.values())} in {ANSWERS_DIR}")
print(f"Reproduction: {REPRODUCED} of {len(reproduction)} originals identical to Experiment 1")
display(pd.DataFrame(by_type).T)
print("\nDone. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units.")

Saved and checked: Qwen-SEA-LION-v4-8B-VL.jsonl, Qwen-SEA-LION-v4-8B-VL.csv, Qwen-SEA-LION-v4-8B-VL_reproduction.csv, Qwen-SEA-LION-v4-8B-VL_run.json in /content/drive/MyDrive/CS612_stage1/answers
Reproduction: 200 of 200 originals identical to Experiment 1


,answers,empty,median_words,min_words,max_words,hit_token_cap,refusal_like,edited_for_excel
original,200.0,0.0,128.0,98.0,163.0,0.0,0.0,0.0
typo,200.0,0.0,127.0,103.0,162.0,0.0,0.0,0.0
synonym,200.0,0.0,130.0,102.0,169.0,0.0,0.0,0.0
restructure,200.0,0.0,129.0,106.0,162.0,0.0,0.0,0.0
paraphrase,200.0,0.0,129.0,98.0,157.0,0.0,1.0,0.0



Done. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units.
